# MATH 60231 – Homework #4
**Group:** Mathis Huot, Nathan Madore, Isabelle Tanara

Due: Monday, November 30, 2026

# Setup

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm
from scipy import stats

## Table formatting functions

Tables are drawn in the style of finance research papers (horizontal rules only, serif font, optional panels and notes). The functions are reused from our own Case Study 1 notebook. The underlying matplotlib table/text drawing follows [MatPlotLib - Table Demo](https://matplotlib.org/stable/gallery/misc/table_demo.html).

In [2]:
# Table formatting (reused from our Case Study 1 notebook, adapted to one Output/ folder)
import os, re, textwrap
from IPython.display import display
os.makedirs("Output", exist_ok=True)                       # every table and figure is also saved as a PNG in Output/


def save_output(fig, label):
    """Save a matplotlib figure as a PNG (200 dpi) in the Output/ folder.

    Parameters
    ----------
    fig : matplotlib.figure.Figure
        Figure to save.
    label : str
        Table/figure title; turned into a safe file name
        (e.g. "Table 2.1 - Descriptive statistics" -> "Output/Table_2.1_Descriptive_statistics.png").
    """
    name = re.sub(r"[^A-Za-z0-9.]+", "_", label).strip("_.")[:90]
    fig.savefig(f"Output/{name}.png", dpi=200, bbox_inches="tight")


# ---------- Number formats used in the tables ----------
#   "num"  -> 2 decimals            "num3" -> 3 decimals (e.g. autocorrelations)
#   "num4" -> 4 decimals            "pval" -> 3 decimals (p-values)
#   "pct"  -> 0.0013 shown as 0.13% "int"  -> 397

def format_value(x, kind="num"):
    """Format one number as text for a table.

    Parameters
    ----------
    x : float or str
        Value to format (NaN is shown as a dash, text such as a date is shown as is).
    kind : str
        "num" (2 decimals), "num3" (3 decimals), "num4" (4 decimals), "pval" (3 decimals),
        "pct" (percentage, 2 decimals) or "int" (whole number).

    Returns
    -------
    str
        Formatted value, e.g. 0.0013 with kind="pct" -> "0.13%".
    """
    if isinstance(x, str):
        return x
    if pd.isna(x):
        return "–"
    if kind == "pct":
        return f"{x * 100:.2f}%"
    if kind == "int":
        return f"{x:,.0f}"
    if kind in ("pval", "num3"):
        return f"{x:,.3f}"
    if kind == "num4":
        return f"{x:,.4f}"
    return f"{x:,.2f}"


def format_table(df, row_formats=None, col_formats=None, default="num"):
    """Return a copy of a table with every number formatted as text.

    The format of a cell is taken from its column (col_formats) if listed, otherwise from
    its row (row_formats), otherwise `default`.

    Parameters
    ----------
    df : pd.DataFrame
        Table of numbers.
    row_formats : dict, optional
        Maps a row label to a format kind (see format_value).
    col_formats : dict, optional
        Maps a column label to a format kind (useful when rows are series and columns are statistics).
    default : str
        Format kind used when neither the row nor the column is listed.

    Returns
    -------
    pd.DataFrame
        Same shape as df, with formatted strings.
    """
    row_formats, col_formats = row_formats or {}, col_formats or {}
    out = df.astype(object).copy()
    for r in df.index:
        for c in df.columns:
            kind = col_formats.get(c, row_formats.get(r, default))
            out.loc[r, c] = format_value(df.loc[r, c], kind)
    return out


def split_title(title):
    """Split "Table 2.1 - Description" (hyphen or en dash) into ("Table 2.1", "Description")."""
    number, _, description = re.sub(r"\s+[–-]\s+", " - ", title, count=1).partition(" - ")
    return number, description


# ---------- Table image in the style of finance research papers ----------
def save_table_png(panels, title, notes=None, row_formats=None, col_formats=None, default="num",
                   width=7.5, label_width=3.3):
    """Draw a table as an image in the style of finance research papers and save it to Output/.

    Horizontal rules only (thick top and bottom, thin under the header and between panels),
    italic panel labels, serif font, and an optional notes paragraph under the table.

    Parameters
    ----------
    panels : dict
        {panel label: DataFrame}; all panels must have the same columns ("" = no panel label).
    title : str
        "Table X.Y - Description" (number in bold, description below).
    notes : str, optional
        Notes printed under the table.
    row_formats, col_formats, default
        Number formats (see format_table).
    width : float
        Width of the image in inches.
    label_width : float
        Width of the row-label column in inches.
    """
    columns = list(next(iter(panels.values())).columns)

    # List of lines to draw, top to bottom
    lines = []
    for panel_name, df in panels.items():
        if panel_name:
            lines.append(("panel", panel_name))
        for label, row in format_table(df, row_formats, col_formats, default).iterrows():
            lines.append(("row", label, list(row)))

    # Sizes in inches (the drawing uses inches as coordinates, measured from the top)
    margin, row_h = 0.3, 0.26
    title_h = 0.65
    notes_text = textwrap.fill(notes, int(150 * width / 7.5)) if notes else ""
    notes_h = 0.16 * (notes_text.count("\n") + 1) + 0.2 if notes else 0.1
    table_h = row_h * (len(lines) + 1.6) + 0.12 * (len(panels) - 1)
    height = title_h + table_h + notes_h

    left, right = margin, width - margin
    first_value_x = margin + label_width                 # values start to the right of the row labels
    col_w = (right - first_value_x) / len(columns)
    col_x = [first_value_x + (i + 0.5) * col_w for i in range(len(columns))]

    with plt.rc_context({"font.family": "serif", "font.serif": ["Times New Roman", "Times", "DejaVu Serif"]}):
        fig = plt.figure(figsize=(width, height))
        ax = fig.add_axes([0, 0, 1, 1])
        ax.set_xlim(0, width)
        ax.set_ylim(height, 0)                           # y grows downward
        ax.axis("off")

        number, description = split_title(title)
        ax.text(width / 2, 0.2, number, ha="center", va="center", fontsize=11, fontweight="bold")
        ax.text(width / 2, 0.45, description, ha="center", va="center", fontsize=11)

        y = title_h
        ax.hlines(y, left, right, color="black", linewidth=1.3)              # top rule
        y += row_h * 0.6
        for x, col in zip(col_x, columns):
            ax.text(x, y, col, ha="center", va="center", fontsize=10)
        y += row_h * 0.55
        ax.hlines(y, left, right, color="black", linewidth=0.6)              # rule under the header
        y += row_h * 0.55

        first_panel = True
        for line in lines:
            if line[0] == "panel":
                if not first_panel:                                          # thin rule between panels
                    y += 0.06
                    ax.hlines(y - row_h * 0.5, left, right, color="black", linewidth=0.4)
                    y += 0.06
                ax.text(left, y, line[1], ha="left", va="center", fontsize=10, style="italic")
                first_panel = False
            else:
                _, label, values = line
                ax.text(left + 0.15, y, label, ha="left", va="center", fontsize=10)
                for x, value in zip(col_x, values):
                    ax.text(x, y, value, ha="center", va="center", fontsize=10)
            y += row_h

        y -= row_h * 0.45
        ax.hlines(y, left, right, color="black", linewidth=1.3)              # bottom rule

        if notes:
            ax.text(left, y + 0.12, notes_text, ha="left", va="top", fontsize=8, linespacing=1.4)

        save_output(fig, title)
        plt.close(fig)   # saved only; the formatted table is displayed in the notebook instead


def save_table(table, title, notes=None, row_formats=None, col_formats=None, default="num", **layout):
    """Save a table as a finance-paper-style PNG in Output/ and display it formatted in the notebook.

    Parameters
    ----------
    table : pd.DataFrame or dict
        One table, or {"Panel A: ...": df_a, "Panel B: ...": df_b} for a multi-panel table.
    title : str
        "Table X.Y - Description"; also used for the file name.
    notes : str, optional
        Notes printed under the table.
    row_formats, col_formats, default
        Number formats (see format_table).
    **layout
        Optional width and label_width of the image (see save_table_png).
    """
    panels = table if isinstance(table, dict) else {"": table}
    save_table_png(panels, title, notes, row_formats, col_formats, default, **layout)

    print(title)
    for panel_name, df in panels.items():
        if panel_name:
            print(panel_name)
        display(format_table(df, row_formats, col_formats, default))

## Data loading

Monthly data from January 1980 to December 2021, taken from Prof. Ken French's Data Library.

In [3]:
FF_data = pd.read_csv("Data/F-F_Research_Data_5_Factors_2x3.csv", skiprows=3, index_col=0)
FF_data = FF_data.loc["198001":"202112"].astype(float)
FF_data.index = pd.to_datetime(FF_data.index, format="%Y%m").to_period("M")

FF_momentum = pd.read_csv("Data/F-F_Momentum_Factor.CSV", skiprows=13, index_col=0)
FF_data = FF_data.loc["198001":"202112"].astype(float)

Industry_pf = pd.read_csv("Data/17_Industry_Portfolios.CSV", skiprows = 11, index_col=0)
Industry_pf = Industry_pf[~Industry_pf.index.duplicated(keep="first")]
Industry_pf = Industry_pf.loc["198001":"202112"].astype(float)

# Problem 1 – Tests of Asset Pricing Models [50 pts]
Monthly data, January 1980 – December 2021, from Prof. Ken French's Data Library.

## 1a. `Q1_KenFrenchData()` [10 pts]
Function with no arguments returning `factors` (Mkt-RF, SMB, HML, RMW, CMA from *F-F_Research_Data_5_Factors_2x3*, then Mom from *F-F_Momentum_Factor*) and `portfolios` (17 Industry Portfolios). All series must be **excess returns** (net of RF). Report and interpret the first five rows and descriptive statistics of each object.

In [4]:
def read_french_monthly(path): # Read the first monthly table of a Ken French CSV, Jan 1980 - Dec 2021.
    with open(path) as f:
        lines = f.readlines()
    # header row is the first line starting with a comma; the table ends at the next blank line
    start = next(i for i, line in enumerate(lines) if line.lstrip().startswith(","))
    end = next((i for i in range(start + 1, len(lines)) if lines[i].strip() == ""), len(lines))

    df = pd.read_csv(path, skiprows=start, nrows=end - start - 1, index_col=0)
    df.columns = df.columns.str.strip()
    df.index = pd.to_datetime(df.index.astype(str).str.strip(), format="%Y%m").to_period("M")
    return df.loc["1980-01":"2021-12"]

In [5]:
def Q1_KenFrenchData():
    ff5 = read_french_monthly("Data/F-F_Research_Data_5_Factors_2x3.csv")
    mom = read_french_monthly("Data/F-F_Momentum_Factor.csv")
    ind = read_french_monthly("Data/17_Industry_Portfolios.csv")

    factors = pd.concat([ff5[["Mkt-RF", "SMB", "HML", "RMW", "CMA"]], mom["Mom"]], axis=1)
    # Excess Returns (Net of the Risk-Free Rate) (Factors already reflect this)
    portfolios = ind.sub(ff5["RF"], axis=0)   # Industry returns minus the risk-free rate
    return factors, portfolios

factors, portfolios = Q1_KenFrenchData()

In [6]:
factors.describe()

,Mkt-RF,SMB,HML,RMW,CMA,Mom
count,504.000000,504.000000,504.000000,504.000000,504.000000,504.000000
mean,0.739345,0.100179,0.195417,0.379742,0.253710,0.535377
std,4.482055,2.910361,3.057619,2.402923,1.987874,4.483515
min,-23.240000,-15.320000,-13.880000,-18.650000,-6.800000,-34.300000
25%,-1.920000,-1.590000,-1.540000,-0.760000,-1.005000,-1.142500
50%,1.170000,0.010000,0.085000,0.335000,0.065000,0.630000
75%,3.492500,1.875000,1.750000,1.362500,1.437500,2.885000
max,13.650000,18.280000,12.470000,13.070000,9.070000,18.200000


In [7]:
portfolios.describe()

,Food,Mines,Oil,Clths,Durbl,Chems,Cnsum,Cnstr,Steel,FabPr,Machn,Cars,Trans,Utils,Rtail,Finan,Other
count,504.000000,504.000000,504.000000,504.000000,504.000000,504.000000,504.000000,504.000000,504.000000,504.000000,504.000000,504.000000,504.000000,504.000000,504.000000,504.000000,504.000000
mean,0.881091,0.617004,0.644067,0.882361,0.603433,0.738571,0.890238,0.923333,0.527222,0.747540,0.841726,0.955357,0.800516,0.649246,0.945198,0.795377,0.775198
std,4.164380,7.941516,6.390870,6.113349,5.731596,5.852139,4.313624,6.041392,8.180360,5.498653,6.701123,7.164003,5.534051,3.930420,5.099154,5.514916,4.937564
min,-18.800000,-32.850000,-34.790000,-32.040000,-25.900000,-28.510000,-19.700000,-29.840000,-32.490000,-29.150000,-29.370000,-27.990000,-29.660000,-13.070000,-28.540000,-22.440000,-23.400000
25%,-1.302500,-4.567500,-2.785000,-2.160000,-2.627500,-2.772500,-1.802500,-2.837500,-4.290000,-2.065000,-2.900000,-2.977500,-2.077500,-1.572500,-2.077500,-2.370000,-2.202500
50%,0.960000,0.430000,0.770000,0.980000,0.770000,1.030000,1.095000,1.150000,0.590000,1.045000,1.310000,0.790000,1.125000,0.955000,0.800000,1.310000,1.350000
75%,3.392500,5.860000,4.257500,4.665000,4.050000,4.135000,3.610000,4.362500,5.407500,4.010000,4.602500,4.650000,4.112500,3.230000,4.035000,4.095000,3.940000
max,14.810000,22.050000,32.840000,23.920000,29.620000,21.880000,15.380000,18.010000,30.300000,18.930000,19.470000,39.310000,19.940000,11.260000,18.440000,17.090000,14.970000


## 1b. Descriptive statistics & graphs [10 pts]
Graphs (at minimum): (i) time-series plot of the six factors; (ii) time-series plot of the 17 industry excess returns; (iii) correlation heatmap of the factors. Discuss: most volatile factors, industries with highest/lowest mean excess returns, multicollinearity implied by the factor correlations.

## 1c. `Q1_AssetPricingModel()` [20 pts]
Inputs: `Factors` (T×K), `Assets` (T×N), boolean `Traded`. Two-pass estimation: (i) time-series regressions $r_{it} = \alpha_i + \beta_i' F_t + \varepsilon_{it}$; (ii) cross-sectional regression of mean excess returns $\bar r_i$ on $\hat\beta_i$ to get risk premia $\hat\lambda$.

Prints a summary and returns: $\hat\lambda$ (K×1) and robust t-stats (K×1); $\hat\beta$ (N×K) and robust t-stats (N×K); cross-sectional $R^2$; J-test [statistic, p-value, dof] (3×1). Apply with `Traded=True` and `Traded=False`; report and interpret both.

## 1d. Interpretation [10 pts]
Answered in the write-up (2–3 sentences each): significant risk premia under `Traded=True`; J-test conclusion; why the J-test null distribution depends on `Traded`; which version is theoretically appropriate here.

# Problem 2 – Maximum Likelihood Estimation [35 pts]
Dependent variable: Mkt-RF. Regressors: SMB, HML, RMW, CMA, Mom (same 1980–2021 monthly data as Problem 1).

## 2a. `Q2_OLSStudentT()` [8 pts]
Inputs: `df` (first column = dependent variable, others = regressors), integer `dof > 0`. (i) OLS on a constant and the regressors; (ii) residuals $\hat e_t$ and $\hat\sigma$; (iii) log-likelihood assuming $\hat e_t/\hat\sigma$ i.i.d. $t(\text{dof})$. Returns `llf`. Apply with dof = 5 and dof = 30; explain the difference (tail behaviour of the residuals).

In [8]:
def Q2_OLSStudentT(df, dof):
    """
    OLS of df's first column on a constant + the other columns, then the
    log-likelihood assuming the standardized residuals e_t / sigma are i.i.d. t(dof).
    """
    if not (isinstance(dof, (int, np.integer)) and dof > 0):
        raise ValueError("dof must be an integer > 0")

    # (i) OLS of the first column on a constant and the other columns
    y = df.iloc[:, 0]
    X = sm.add_constant(df.iloc[:, 1:])
    res = sm.OLS(y, X, missing="drop").fit()

    # (ii) residuals and their estimated standard deviation
    e = res.resid
    sigma = np.sqrt(res.scale)          # sqrt(SSR / (T - k))

    # (iii) log-likelihood with z_t = e_t / sigma ~ i.i.d. t(dof)
    z = e / sigma
    T = len(e)
    llf = np.sum(stats.t.logpdf(z, df=dof)) - T * np.log(sigma)

    return llf

In [ ]:
q2_df = factors[["Mkt-RF", "SMB", "HML", "RMW", "CMA", "Mom"]].dropna()   # Mkt-RF first = dependent variable

llf_5  = Q2_OLSStudentT(q2_df, 5)
llf_30 = Q2_OLSStudentT(q2_df, 30)

print(f"Log-likelihood, t(5):  {llf_5:.2f}")
print(f"Log-likelihood, t(30): {llf_30:.2f}")

Log-likelihood, t(5):  -1406.77
Log-likelihood, t(30): -1399.58


In [12]:
def llf_std_t(df, dof):
    # unit-variance Student t (requires dof > 2)
    y = df.iloc[:, 0]
    X = sm.add_constant(df.iloc[:, 1:])
    res = sm.OLS(y, X, missing="drop").fit()
    e = res.resid
    sigma = np.sqrt(res.scale)
    z = e / sigma
    c = np.sqrt(dof / (dof - 2))              # rescales t(dof) to variance 1
    T = len(e)
    return np.sum(stats.t.logpdf(z * c, df=dof)) + T * np.log(c) - T * np.log(sigma)

print("Standardized t(5): ", round(llf_std_t(q2_df, 5), 2))
print("Standardized t(30):", round(llf_std_t(q2_df, 30), 2))
print("Normal (OLS llf):  ", round(sm.OLS(q2_df.iloc[:, 0], sm.add_constant(q2_df.iloc[:, 1:])).fit().llf, 2))

Standardized t(5):  -1389.79
Standardized t(30): -1398.22
Normal (OLS llf):   -1404.63


## 2b. `Q2_LRTest()` [9 pts]
Inputs: `df`, integer `dof > 0`. Uses `Q2_OLSStudentT()` to compute the LR statistic for $H_0$: all coefficients except the constant are zero. Returns the LR statistic and its p-value. Apply with dof = 5: (i) restricted model and its log-likelihood vs. unrestricted; (ii) do the five factors jointly predict Mkt-RF?; (iii) does the LR statistic change with dof = 30?

## 2c. `Q2_ProfileLikelihood()` [9 pts]
Input: `df`. Evaluates `Q2_OLSStudentT(df, dof)` at $\text{dof}_j = e^{j/12}$, $j = 0, \dots, 50$, plots the profile likelihood and returns a 51×2 array [dof, llf]. Interpret: (i) shape of the curve; (ii) location of the maximum and implied MLE of dof.

## 2d. MLE of the degrees of freedom [9 pts]
Answered in the write-up from the profile likelihood in 2c: (i) the estimate (≈ 1, ≈ 5, ≈ 15 or > 60); (ii) tail behaviour; (iii) ruling out Cauchy (ν = 1) and near-normal (ν > 60); (iv) consequences for the inference in Problem 1.

# Problem 3 – PCA and Factor Models [15 pts]
No coding required. Answered in the write-up: (a) recent news article and citation; (b) falsifiable hypothesis; (c) PCA design (matrix X, demeaning/standardizing, choice of K); (d) mathematical model ($\Sigma = V\Lambda V'$, $X \approx F V_K'$); (e) conversion to a linear factor model in the spirit of `Q1_AssetPricingModel()`.